<div align="center">
  <img src="../assets/uepb_logo.png" width="150">
  <br>
  <strong>Universidade Estadual da Paraíba (UEPB)</strong><br>
  <strong>Professor(a):</strong> Thiciany Matsudo Iwano<br>
  <strong>Alunos:</strong> Tarcio Elyakin Agra Diniz & Djhonatah Wesley Cavalcanti Alves<br>
  <strong>Série:</strong> Sistemas Lineares: teoria, interpretação geométrica e aplicações com Python<br>
</div>

---

# [Tutorial 08] Prática: Exercícios Resolvidos

> **Pré-requisito:** [07] Aplicação: Interpolação Polinomial (`07_aplicacao_interpolacao_polinomial.ipynb`)  
> **Próximo:** [09] Conclusão e Próximos Passos (`09_conclusao_e_proximos_passos.ipynb`)

---

## Introdução

Chegamos ao momento de consolidar tudo o que aprendemos. A melhor forma de dominar a Álgebra Linear não é apenas lendo a teoria, mas resolvendo problemas reais e comparando diferentes abordagens.

Neste tutorial, vamos enfrentar três tipos de desafios: um sistema clássico resolvido passo a passo, um sistema com infinitas soluções e uma aplicação prática na agronomia. Vamos alternar entre o **raciocínio manual** (essencial para exames e concursos) e a **eficiência do Python** (essencial para o mercado de trabalho).

Ao final deste tutorial, você será capaz de:
1. Resolver sistemas 3x3 manualmente com confiança.
2. Parametrizar soluções de sistemas indeterminados (SPI).
3. Modelar e resolver problemas de mistura de componentes químicos.
4. Escolher a ferramenta certa (Manual, SymPy ou NumPy) para cada situação.

---

## 1. Exercício 1: Sistema 3x3 e a Escada de Gauss (SPD)

**Problema:** Encontre a solução única para o sistema abaixo:

$$
\begin{cases}
x + y + z = 6 \\
2x - y + z = 3 \\
3x + y - 2z = -1
\end{cases}
$$

### 1.1 Resolução Manual (Passo a Passo)

1. **Montar a Matriz Aumentada:**
$$
[A|b] = \begin{pmatrix}
1 & 1 & 1 & | & 6 \\
2 & -1 & 1 & | & 3 \\
3 & 1 & -2 & | & -1
\end{pmatrix}
$$

2. **Zerar a primeira coluna abaixo do pivô $a_{11}=1$:**
   - $L_2 \leftarrow L_2 - 2L_1$: 
     $2 - 2(1) = 0$; $-1 - 2(1) = -3$; $1 - 2(1) = -1$; $3 - 2(6) = -9$
   - $L_3 \leftarrow L_3 - 3L_1$: 
     $3 - 3(1) = 0$; $1 - 3(1) = -2$; $-2 - 3(1) = -5$; $-1 - 3(6) = -19$
   
   Nova matriz:
$$
\begin{pmatrix}
1 & 1 & 1 & | & 6 \\
0 & -3 & -1 & | & -9 \\
0 & -2 & -5 & | & -19
\end{pmatrix}
$$

3. **Zerar a segunda coluna abaixo do pivô $a_{22}=-3$:**
   - Para facilitar, podemos fazer $L_2 \leftarrow -\frac{1}{3}L_2$ ou usar frações. Vamos usar $L_3 \leftarrow L_3 - \frac{2}{3}L_2$:
     $-2 - \frac{2}{3}(-3) = 0$; $-5 - \frac{2}{3}(-1) = -5 + \frac{2}{3} = -\frac{13}{3}$; $-19 - \frac{2}{3}(-9) = -19 + 6 = -13$
   
   Matriz na forma escalonada (REF):
$$
\begin{pmatrix}
1 & 1 & 1 & | & 6 \\
0 & -3 & -1 & | & -9 \\
0 & 0 & -13/3 & | & -13
\end{pmatrix}
$$

4. **Substituição Regressiva:**
   - Da $L_3$: $-\frac{13}{3}z = -13 \implies z = 3$
   - Da $L_2$: $-3y - z = -9 \implies -3y - 3 = -9 \implies -3y = -6 \implies y = 2$
   - Da $L_1$: $x + y + z = 6 \implies x + 2 + 3 = 6 \implies x = 1$

**Solução:** $(1, 2, 3)$.

---

### 1.2 Resolução Computacional (SymPy)

Vamos usar o SymPy para validar o processo simbólico.


In [1]:
import sympy as sp

# Definindo variáveis
x, y, z = sp.symbols('x y z')

# Matriz aumentada
M1 = sp.Matrix([
    [1, 1, 1, 6],
    [2, -1, 1, 3],
    [3, 1, -2, -1]
])

# Obtendo a RREF
rref_M1, pivots = M1.rref()

print("Matriz em forma RREF (SymPy):")
sp.pprint(rref_M1)

# Solução direta
sol1 = sp.solve_linear_system(M1, x, y, z)
print(f"\nSolução encontrada: {sol1}")


Matriz em forma RREF (SymPy):
⎡1  0  0  1⎤
⎢          ⎥
⎢0  1  0  2⎥
⎢          ⎥
⎣0  0  1  3⎦

Solução encontrada: {x: 1, y: 2, z: 3}


---

## 2. Exercício 2: Sistema com Variáveis Livres (SPI)

**Problema:** Analise e resolva o sistema:

$$
\begin{cases}
x_1 + 2x_2 - x_3 + x_4 = 2 \\
2x_1 + 4x_2 + x_3 - 3x_4 = 5
\end{cases}
$$

### 2.1 Resolução Manual

1. **Matriz Aumentada:**
$$
\begin{pmatrix}
1 & 2 & -1 & 1 & | & 2 \\
2 & 4 & 1 & -3 & | & 5
\end{pmatrix}
$$

2. **Zerar abaixo de $a_{11}$:** $L_2 \leftarrow L_2 - 2L_1$
   $2-2=0$; $4-4=0$; $1-(-2)=3$; $-3-2=-5$; $5-4=1$

$$
\begin{pmatrix}
1 & 2 & -1 & 1 & | & 2 \\
0 & 0 & 3 & -5 & | & 1
\end{pmatrix}
$$

**Análise:** Temos 4 variáveis e apenas 2 pivôs (nas colunas 1 e 3).
- Variáveis líderes: $x_1, x_3$
- Variáveis livres: $x_2, x_4$

3. **Expressar as líderes em função das livres:**
   - Da $L_2$: $3x_3 - 5x_4 = 1 \implies x_3 = \frac{1 + 5x_4}{3}$
   - Da $L_1$: $x_1 = 2 - 2x_2 + x_3 - x_4 = 2 - 2x_2 + \frac{1 + 5x_4}{3} - x_4$
     $x_1 = \frac{6 - 6x_2 + 1 + 5x_4 - 3x_4}{3} = \frac{7 - 6x_2 + 2x_4}{3}$

### 2.2 Verificação com SymPy


In [2]:
x1, x2, x3, x4 = sp.symbols('x1 x2 x3 x4')
M2 = sp.Matrix([
    [1, 2, -1, 1, 2],
    [2, 4, 1, -3, 5]
])

sol2 = sp.solve_linear_system(M2, x1, x2, x3, x4)
print("Solução Geral (SPI):")
print(sol2)


Solução Geral (SPI):
{x1: -2*x2 + 2*x4/3 + 7/3, x3: 5*x4/3 + 1/3}


---

## 3. Exercício 3: Aplicação Prática - Mistura de Nutrientes

**Cenário:** Um agrônomo precisa criar 100 kg de um fertilizante que contenha 12% de Nitrogênio (N), 10% de Fósforo (P) e 8% de Potássio (K). Ele tem três fertilizantes base disponíveis:
- **Base A:** 20% N, 10% P, 5% K
- **Base B:** 10% N, 15% P, 10% K
- **Base C:** 5% N, 5% P, 15% K

Quanto de cada base ($x, y, z$) ele deve usar?

### 3.1 Montagem do Sistema

1.  **Massa Total:** $x + y + z = 100$
2.  **Balanço de Nitrogênio:** $0.20x + 0.10y + 0.05z = 0.12(100) = 12$
3.  **Balanço de Fósforo:** $0.10x + 0.15y + 0.05z = 0.10(100) = 10$
4.  **Balanço de Potássio (Opcional/Verificação):** $0.05x + 0.10y + 0.15z = 0.08(100) = 8$

Temos 4 equações para 3 incógnitas. Vamos usar as três primeiras para resolver e a quarta para validar.

### 3.2 Resolução com NumPy (Eficiência Numérica)

Em problemas reais com dados decimais, `numpy.linalg.solve` é a escolha padrão.


In [3]:
import numpy as np

# Coeficientes das 3 primeiras equações
A_nutri = np.array([
    [1.0, 1.0, 1.0],      # Massa total
    [0.20, 0.10, 0.05],   # N
    [0.10, 0.15, 0.05]    # P
])

b_nutri = np.array([100, 12, 10])

try:
    sol_nutri = np.linalg.solve(A_nutri, b_nutri)
    print("Quantidades necessárias (kg):")
    print(f"Base A: {sol_nutri[0]:.2f}")
    print(f"Base B: {sol_nutri[1]:.2f}")
    print(f"Base C: {sol_nutri[2]:.2f}")
    
    # Validação com a 4ª equação (Potássio)
    k_calculado = 0.05*sol_nutri[0] + 0.10*sol_nutri[1] + 0.15*sol_nutri[2]
    print(f"\nValidação Potássio (K): Esperado 8.0, Obtido {k_calculado:.2f}")

except np.linalg.LinAlgError:
    print("O sistema não possui solução única.")


Quantidades necessárias (kg):
Base A: 36.00
Base B: 32.00
Base C: 32.00

Validação Potássio (K): Esperado 8.0, Obtido 9.80


---

## 4. Comparação: Manual vs Computacional

| Aspecto | Resolução Manual | Resolução Computacional |
| :--- | :--- | :--- |
| **Escopo** | Problemas pequenos (2x2, 3x3). | Problemas de qualquer escala. |
| **Precisão** | Alta (frações exatas). | Numérica (float) ou Simbólica (SymPy). |
| **Intuição** | Essencial para aprender o "porquê". | Focada no resultado e aplicação. |
| **Erros** | Propensa a falhas aritméticas bobas. | Imune a erros de conta, mas sensível a erros de modelagem. |

### Quando usar o quê?
- **Manual:** Estudo conceitual, provas acadêmicas e validação de pequenos passos lógicos.
- **SymPy:** Quando você precisa da resposta exata (fração, raiz, variáveis) e quer que o computador faça a álgebra por você.
- **NumPy:** Quando você lida com dados reais, matrizes grandes ou problemas de engenharia onde a velocidade e a aproximação decimal são suficientes.

---

## 5. Exercícios Extras para o Leitor

1.  **Desafio Geométrico:** Determine a interseção de três planos no espaço $\mathbb{R}^3$. O que acontece se o sistema for SI?
2.  **Rede Elétrica:** Monte as equações de Kirchhoff para um circuito com duas malhas e resolva-o usando a matriz aumentada.
3.  **Ajuste de Parábola:** Encontre a parábola $y = ax^2 + bx + c$ que passa pelos pontos $(1, 4), (2, 0)$ e $(3, 12)$. (Dica: Use a lógica do Notebook 07).

---

## 6. Conclusão

Neste notebook, fechamos o ciclo básico de sistemas lineares. Vimos que, embora a teoria seja construída sobre passos manuais de escalonamento, o domínio das ferramentas computacionais permite transpor esses conceitos para situações reais e complexas.

A Álgebra Linear não é apenas sobre "resolver para x", mas sobre entender como o espaço se organiza e como as relações entre variáveis podem ser manipuladas de forma eficiente.


---
## Próximo Passo

Parabéns! Você concluiu a parte prática intensiva do nosso tutorial. Vimos que a Álgebra Linear é uma ferramenta versátil, capaz de resolver desde problemas acadêmicos abstratos até desafios reais de engenharia e agronomia.

No nosso último encontro, vamos fazer uma síntese de toda a jornada e apontar o caminho para quem deseja se tornar um mestre em Álgebra Linear Computacional.

👉 **[Ir para o Tutorial 09: Conclusão e Próximos Passos](./09_conclusao_e_proximos_passos.ipynb)**
